In [2]:
import pandas as pd
import numpy as np
import os
import csv
# By Virginia Wang 

"""
TO CREATE A NEW MASTER VCF

This file will take in the old master vcf and a folder containing new files to add and append the new information to a NEW FILE
given additional information about the background, condition, instructor, and year. The intent is that the user will rerun this
file per "batch" of background, condition, instructor, and year files. Refer to the comments for additional instructions or 
follow the ones below:

1) Adjust path to the folder containing the raw vcf files that you would like to append
2) Check that the old master vcf and gene info is named correctly (takes from working directory and adds the filename as seen below)
3) Adjust path to output location as well as output file name 
4) Adjust background, condition, instructor, and year names 
5) Run all cells
6) Check output to make sure everything was appended correctly 
7) REPLACE OLD MASTER VCF and change filename accordingly 

*NOTE: sample name will be taken from filename (anything before "_S" appears in filename) ex: D15_S17_samtools_filtered.vcf -> sample = D15
if sample is not found, it will print "sample?" in its place
*NOTE: This program assumes the following information
    a) the .txt raw vcf files will have 47 lines of (unnecessary) header information 
    b) the .txt raw vcf files have the same format as given samples (CHROM POS ID ...)
    c) the gene can be found in the gene_info vcf given the region 
"""
#####################################################################
## %% TO DO: YOU SHOULD CHANGE THIS SECTION AND THIS SECTION ONLY   #
## TO DO: Replace with path to folder with your data.               #
path_FOLDER = r"/Users/leahanderson/Documents/Research/dunham/freeze_thaw_evolution/jan2024_FTevo/VCFs"    #
                                                                    #
## TO DO: Change output path as necessary. Example shown below.     #
path_OUTPUT = r"/Users/leahanderson/Documents/Research/dunham/freeze_thaw_evolution/jan2024_FTevo/VCFs/append_test_240523"         #
output_name = "new_master_vcf.csv"                                      #
                                                                    #
## TO DO: type in necessary information below                       #
background = "" #if empty, it will default to S288c                 #
condition = ""                                                      #
instructor = ""                                                     #
year = ""                                                           #
# sample = (based on file name later below)                         #
                                                                    #
# path to master vcf to add to (change filename/path if needed)     #
path_MASTER = os.path.join(os.getcwd(), "/Users/leahanderson/Documents/GitHub/GSHackathon/all_yEvo_vcf.csv")         #
                                                                    #
# path to gene info csv to match with region                        #
path_GENEINFO = os.path.join(os.getcwd(),"/Users/leahanderson/Documents/GitHub/GSHackathon/gene_info.csv")           #
                                                                    #
#####################################################################



In [ ]:
# Grabbing master vcf (to change) and gene_info (for matching)
master_yEvo = pd.read_csv(path_MASTER)
gene_info = pd.read_csv(path_GENEINFO)

# Iterate through vcf files in the folder
for filename in os.listdir(path_FOLDER):
    # Construct the full file path
    file_path = os.path.join(path_FOLDER, filename)
    # Open individual files
    with open(file_path, 'r') as txt_file:        
        # Skip to the 48th line to grab necessary data
        for _ in range(47):
            next(txt_file)
        # Read the 48th line and strip any leading/trailing whitespaces, split into array by tabs
        data = txt_file.readline().strip().split('\t')
        
        # Grab region
        region = data[11]

        # Search to find corresponding gene in gene_info
        if region == "NA":
            gene = "NA"
        else:
            matching_rows = gene_info[gene_info.iloc[:, 0].str.contains(region, case=False, na=False)]
            gene = matching_rows.iloc[:, 1].values[0]

        # Extract everything before "_S" for sample name
        sample = filename[:filename.find('_S')] if filename.find('_S') != -1 else "sample?"

        CHROM = data[0]
        POS = data[1]
        ID = data[2]
        REF = data[3]
        ALT = data[4]
        QUAL = data[5]
        FILTER = data[6]
        INFO = data[7]
        ANNOTATION = data[10]
        REGION = region
        GENE = gene
        PROTEIN = data[12]
        seq_file = filename
        if background == "":
            background = "S288c"

        # add to dataframe
        columns = [CHROM, POS, ID, REF, ALT, QUAL, FILTER, INFO, ANNOTATION, REGION, GENE, PROTEIN, seq_file, background, condition, instructor, year, sample]
        master_yEvo.loc[len(master_yEvo)] = columns

# save dataframe as csv w/ output name in output path 
master_yEvo.to_csv(os.path.join(path_OUTPUT, output_name), index=False)